<a href="https://colab.research.google.com/github/Shreyash-Singh-1/Summer_Internship/blob/main/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import plotly.express as px


In [ ]:
df = pd.read_csv("/content/sample_data/train.csv")
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
for col in df.columns:
    print(f"Column '{col}': {df[col].nunique()} unique values")
    print("="*50)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

df = pd.read_csv('train.csv')

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y', errors='coerce')
df['YearMonth'] = df['Order Date'].dt.to_period('M').astype(str)
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce').fillna(0)

# Add 'Segment' for even more granular filtering
regions = ['All'] + sorted(df['Region'].dropna().unique().tolist())
categories = ['All'] + sorted(df['Category'].dropna().unique().tolist())
segments = ['All'] + sorted(df['Segment'].dropna().unique().tolist())


style = {'description_width': 'initial'}
region_drop = widgets.Dropdown(options=regions, value='All', description='🌍 Region:', style=style)
category_drop = widgets.Dropdown(options=categories, value='All', description='📦 Category:', style=style)
segment_drop = widgets.Dropdown(options=segments, value='All', description='👥 Segment:', style=style)

# Container for the dashboard content
dashboard_output = widgets.Output()

def create_kpi_card(title, value, icon):
    """Generates an aesthetic HTML card for a single KPI"""
    return f"""
    <div style="background-color: #ffffff; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);
                padding: 20px; margin: 10px; flex: 1; text-align: center; border-top: 4px solid #1f77b4;">
        <h4 style="margin: 0; color: #555; font-size: 14px; text-transform: uppercase;">{icon} {title}</h4>
        <h2 style="margin: 10px 0 0 0; color: #2c3e50; font-size: 28px;">{value}</h2>
    </div>
    """

def update_dashboard(region, category, segment):
    with dashboard_output:
        clear_output(wait=True)

        # --- Apply Filters ---
        filtered_df = df.copy()
        if region != 'All':
            filtered_df = filtered_df[filtered_df['Region'] == region]
        if category != 'All':
            filtered_df = filtered_df[filtered_df['Category'] == category]
        if segment != 'All':
            filtered_df = filtered_df[filtered_df['Segment'] == segment]

        if filtered_df.empty:
            display(HTML("<h3 style='color:red;'>No data available for these filters.</h3>"))
            return

        # Calculate Enhanced KPIs
        total_sales = filtered_df['Sales'].sum()
        total_orders = filtered_df['Order ID'].nunique()
        total_customers = filtered_df['Customer ID'].nunique()
        aov = total_sales / total_orders if total_orders > 0 else 0

        # Render KPI Banner
        kpi_html = f"""
        <div style="display: flex; justify-content: space-between; background-color: #f4f6f9; padding: 10px; border-radius: 10px;">
            {create_kpi_card("Total Revenue", f"${total_sales:,.0f}", "💰")}
            {create_kpi_card("Total Orders", f"{total_orders:,}", "🛒")}
            {create_kpi_card("Avg Order Value", f"${aov:,.2f}", "📈")}
            {create_kpi_card("Unique Customers", f"{total_customers:,}", "🧑‍🤝‍🧑")}
        </div>
        """
        display(HTML(kpi_html))

        theme = 'plotly_white'

        # 1. Revenue Trend (Area Chart for better aesthetic)
        trend_df = filtered_df.groupby('YearMonth', as_index=False)['Sales'].sum().sort_values('YearMonth')
        fig_trend = px.area(trend_df, x='YearMonth', y='Sales', title='📅 Monthly Revenue Trend',
                            markers=True, template=theme, color_discrete_sequence=['#1f77b4'])
        fig_trend.update_layout(height=350, margin=dict(l=20, r=20, t=40, b=20))

        # 2. Top Performing Products
        top_products = filtered_df.groupby('Product Name', as_index=False)['Sales'].sum().nlargest(10, 'Sales')
        fig_prod = px.bar(top_products, x='Sales', y='Product Name', orientation='h',
                          title='🏆 Top 10 Products by Revenue', template=theme,
                          color='Sales', color_continuous_scale='Blues')
        fig_prod.update_layout(yaxis={'categoryorder':'total ascending'}, height=350, showlegend=False, margin=dict(l=20, r=20, t=40, b=20))

        # 3. Top States
        top_states = filtered_df.groupby('State', as_index=False)['Sales'].sum().nlargest(10, 'Sales')
        fig_states = px.bar(top_states, x='State', y='Sales',
                            title='📍 Top 10 States by Revenue', template=theme,
                            color_discrete_sequence=['#ff7f0e'])
        fig_states.update_layout(height=350, margin=dict(l=20, r=20, t=40, b=20))

        # 4. Sales Distribution by Ship Mode (Donut)
        ship_df = filtered_df.groupby('Ship Mode', as_index=False)['Sales'].sum()
        fig_ship = px.pie(ship_df, values='Sales', names='Ship Mode', hole=0.5,
                          title='🚚 Revenue by Ship Mode', template=theme,
                          color_discrete_sequence=px.colors.qualitative.Pastel)
        fig_ship.update_layout(height=350, margin=dict(l=20, r=20, t=40, b=20))


        chart1 = go.FigureWidget(fig_trend)
        chart2 = go.FigureWidget(fig_prod)
        chart3 = go.FigureWidget(fig_states)
        chart4 = go.FigureWidget(fig_ship)


        row1 = widgets.HBox([chart1])
        row2 = widgets.HBox([chart2, chart3])
        row3 = widgets.HBox([chart4])

        display(widgets.VBox([row1, row2, row3]))

widgets.interactive_output(update_dashboard, {
    'region': region_drop,
    'category': category_drop,
    'segment': segment_drop
})

# Display the control panel at the top
header = HTML("<h2 style='text-align:center; color:#2c3e50;'>📊 Executive Sales & Revenue Dashboard</h2>")
controls = widgets.HBox([region_drop, category_drop, segment_drop], layout=widgets.Layout(justify_content='center', padding='10px'))

# Render everything
display(header, controls, dashboard_output)

# Trigger initial render
update_dashboard('All', 'All', 'All')